# 02 - Preparation (Phases 3 and 4a)

Ligands from SMILES, receptors from one shared recipe.

Two rules carried from `notes/decisions.md`:

1. **Ligands are built from SMILES, never from crystal coordinates.** Handing the docking
   program part of the answer would make the redocking test in notebook 03 meaningless.
2. **The cofactor stays in the receptor.** Trimethoprim binds alongside NADPH/NADP+, not
   instead of it. Removing it frees the cofactor's space and every system scores well,
   including human - the most common DHFR docking error.

Kernel: **Python (dhfr)**.

In [1]:
import sys

sys.path.insert(0, "../scripts")

import pandas as pd

import dhfr_lib as D

PY = sys.executable
D.toolchain()

{'rdkit': '2026.09.2',
 'biopython': '1.88',
 'meeko': '0.8.0',
 'vina': 'AutoDock Vina v1.2.7'}

## Ligands

Protonation is the judgement call that matters most. Trimethoprim's diaminopyrimidine
ring is substantially protonated at physiological pH, and the protonated form is what
Asp27 grips. Both forms are built: protonated for the main runs, neutral as the
comparison.

In [2]:
LIGANDS = {
    "tmp_protonated": D.TMP_SMILES_PROTONATED,
    "tmp_neutral": D.TMP_SMILES,
    "mtx": D.MTX_SMILES,
}

rows = []
for name, smi in LIGANDS.items():
    path = D.prepare_ligand(smi, name)
    rows.append({"ligand": name, "file": path.name,
                 "rotatable_branches": D.n_rotatable(path), "smiles": smi})

ligand_table = pd.DataFrame(rows)
ligand_table[["ligand", "file", "rotatable_branches"]]

,ligand,file,rotatable_branches
0,tmp_protonated,tmp_protonated.pdbqt,7
1,tmp_neutral,tmp_neutral.pdbqt,7
2,mtx,mtx.pdbqt,13


In [3]:
from rdkit import Chem
from rdkit.Chem import Descriptors

for name, smi in LIGANDS.items():
    m = Chem.MolFromSmiles(smi)
    print(f"{name:16s} heavy={m.GetNumAtoms():3d}  MW={Descriptors.MolWt(m):7.2f}  "
          f"formula={Chem.rdMolDescriptors.CalcMolFormula(m)}")

print()
print("Note: the project plan expects 'roughly five to six' rotatable bonds for")
print("trimethoprim. Meeko reports 7 - three methoxy torsions, the methylene bridge")
print("and the ring-bridge bond. 7 is correct; the plan's range is wrong.")

tmp_protonated   heavy= 21  MW= 291.33  formula=C14H19N4O3+
tmp_neutral      heavy= 21  MW= 290.32  formula=C14H18N4O3
mtx              heavy= 33  MW= 454.45  formula=C20H22N8O5

Note: the project plan expects 'roughly five to six' rotatable bonds for
trimethoprim. Meeko reports 7 - three methoxy torsions, the methylene bridge
and the ring-bridge bond. 7 is correct; the plan's range is wrong.


## Receptors

One recipe for all four targets: keep chain A, drop waters and crystallisation
additives, keep the cofactor, drop the co-crystallised ligand, resolve altlocs to A,
delete residues too incomplete to match a template.

Using the identical recipe everywhere is what makes the cross-system comparison mean
anything.

In [4]:
# name -> (source, co-crystallised ligand to remove)
TARGETS = {
    "2w9g": (D.fetch_pdb("2W9G"), ("TOP",)),   # S. aureus + TMP + NADPH - redock gate
    "1rx2": (D.fetch_pdb("1RX2"), ("FOL",)),   # E. coli + folate + NADP+
    "1u72": (D.fetch_pdb("1U72"), ("MTX",)),   # human + MTX + NADPH
}

receptors, crystal_ligands = {}, {}

for name, (src, drop) in TARGETS.items():
    rec, ligs = D.prepare_receptor(src, name, drop_ligands=drop, python_bin=PY)
    receptors[name] = rec
    crystal_ligands[name] = ligs
    extracted = {k: len(v) for k, v in ligs.items()}
    print(f"{name}: deleted {rec.deleted or 'none'}, extracted {extracted}")

2w9g: deleted ['A:158'], extracted {'TOP': 21}


1rx2: deleted none, extracted {'FOL': 32}


1u72: deleted none, extracted {'MTX': 33}


In [5]:
# The AlphaFold model is apo - no cofactor, no ligand, nothing to drop.
# That is itself a finding: the prediction cannot supply the NADPH the crystal
# receptors carry, so run 3 is not a like-for-like comparison. Say so in the write-up.
#
# It also arrives in AlphaFold's own coordinate frame, ~51 A from the deposited
# 1RX2 crystal. Superpose first, or the borrowed box lands in empty solvent.
af_raw = D.fetch_alphafold("P0ABQ4")["pdb"]

af_fitted, fit_rms, n_paired = D.superpose_onto(
    af_raw, D.RAW / "1RX2.pdb", D.PREP / "af_ecoli_fitted.pdb"
)
print(f"superposed AlphaFold onto 1RX2: {n_paired} CA pairs, RMSD {fit_rms:.3f} A")

rec_af, _ = D.prepare_receptor(af_fitted, "af_ecoli", keep_cofactor=False, python_bin=PY)
receptors["af_ecoli"] = rec_af
print(f"af_ecoli: deleted {rec_af.deleted or 'none'}")

# Confirm the frames now agree.
import numpy as np

def com(path, chain=None):
    c = [[float(l[30:38]), float(l[38:46]), float(l[46:54])]
         for l in open(path)
         if l.startswith(("ATOM", "HETATM")) and (chain is None or l[21] == chain)]
    return np.array(c).mean(axis=0)

sep = np.linalg.norm(com(D.RAW / "1RX2.pdb", "A") - com(af_fitted))
print(f"centre-of-mass separation after fitting: {sep:.2f} A (was 51.04 A)")

superposed AlphaFold onto 1RX2: 159 CA pairs, RMSD 0.361 A


af_ecoli: deleted none
centre-of-mass separation after fitting: 0.48 A (was 51.04 A)


In [6]:
def pdbqt_stats(path):
    lines = [l for l in path.read_text().splitlines() if l.startswith("ATOM")]
    res = {l[17:20].strip() for l in lines}
    return {
        "atoms": len(lines),
        "has_NADPH": "NDP" in res,
        "has_NADP+": "NAP" in res,
        "additives_left": sorted(res & D.ADDITIVES),
    }


receptor_table = pd.DataFrame([
    {"receptor": n, "file": r.pdbqt.name, "deleted": ",".join(r.deleted) or "-",
     **pdbqt_stats(r.pdbqt)}
    for n, r in receptors.items()
])
receptor_table

,receptor,file,deleted,atoms,has_NADPH,has_NADP+,additives_left
0,2w9g,2w9g_receptor.pdbqt,A:158,1589,True,False,[]
1,1rx2,1rx2_receptor.pdbqt,-,1584,False,True,[]
2,1u72,1u72_receptor.pdbqt,-,1888,True,False,[]
3,af_ecoli,af_ecoli_receptor.pdbqt,-,1531,False,False,[]


## Docking boxes

Centred on the crystal ligand centroid, computed from the extracted atoms - never
guessed. The AlphaFold model has no ligand, so its box is borrowed from `1rx2` after
superposition in notebook 01 (the two share numbering, and the backbone RMSD was
0.361 A, so the site sits in the same place).

In [7]:
boxes = {}
for name, ligs in crystal_ligands.items():
    resname, lines = next(iter(ligs.items()))
    boxes[name] = D.centroid(lines)
    print(f"{name:9s} centred on {resname:4s} -> "
          f"({boxes[name][0]:8.3f}, {boxes[name][1]:8.3f}, {boxes[name][2]:8.3f})")

# Valid only because af_ecoli was superposed onto 1rx2 above.
boxes["af_ecoli"] = boxes["1rx2"]
print(f"{'af_ecoli':9s} borrows the 1rx2 centre (valid after superposition)")

# Verify the borrowed centre actually sits inside the AlphaFold protein.
af_coords = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                      for l in open(af_fitted) if l.startswith("ATOM")])
d = np.linalg.norm(af_coords - boxes["af_ecoli"], axis=1)
print(f"{'':9s} {int((d < 11).sum())} AlphaFold atoms within the 22 A box "
      f"(nearest {d.min():.2f} A)")
assert d.min() < 5, "box centre is not inside the protein - superposition failed"

D.save_json({k: v.tolist() for k, v in boxes.items()}, D.PREP / "boxes.json")
D.save_json(
    {n: {"pdbqt": str(r.pdbqt), "cofactor": r.cofactor, "deleted": r.deleted}
     for n, r in receptors.items()},
    D.PREP / "receptors.json",
)

for name, ligs in crystal_ligands.items():
    for resname, lines in ligs.items():
        (D.PREP / f"{name}_{resname}_xtal.pdb").write_text("\n".join(lines) + "\nEND\n")

print("\nwrote boxes.json, receptors.json and crystal ligand files")

2w9g      centred on TOP  -> (  -5.061,  -31.773,    5.828)
1rx2      centred on FOL  -> (  33.096,   42.342,    7.700)
1u72      centred on MTX  -> (  30.621,   16.876,   -1.624)
af_ecoli  borrows the 1rx2 centre (valid after superposition)
          209 AlphaFold atoms within the 22 A box (nearest 2.90 A)

wrote boxes.json, receptors.json and crystal ligand files


## Pass check 3/4a

In [8]:
checks = {
    "all ligands built": len(ligand_table) == 3,
    "trimethoprim has 7 rotatable branches":
        int(ligand_table.loc[ligand_table.ligand == "tmp_protonated",
                             "rotatable_branches"].iloc[0]) == 7,
    "all receptors built": len(receptors) == 4,
    "cofactor kept in 2w9g": bool(receptor_table.loc[receptor_table.receptor == "2w9g",
                                                     "has_NADPH"].iloc[0]),
    "cofactor kept in 1rx2": bool(receptor_table.loc[receptor_table.receptor == "1rx2",
                                                     "has_NADP+"].iloc[0]),
    "cofactor kept in 1u72": bool(receptor_table.loc[receptor_table.receptor == "1u72",
                                                     "has_NADPH"].iloc[0]),
    "no additives left anywhere":
        all(len(a) == 0 for a in receptor_table["additives_left"]),
}

for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

receptor_table.to_csv(D.RESULTS / "receptor_prep.csv", index=False)
ligand_table.to_csv(D.RESULTS / "ligand_prep.csv", index=False)
print(f"\n{sum(checks.values())}/{len(checks)} passed")

PASS  all ligands built
PASS  trimethoprim has 7 rotatable branches
PASS  all receptors built
PASS  cofactor kept in 2w9g
PASS  cofactor kept in 1rx2
PASS  cofactor kept in 1u72
PASS  no additives left anywhere

7/7 passed
